# Streaming a Zarr Store Into an xarray Dataset

**Purpose**: This notebook demonstrates how to use `hsclient`'s `HydroShare.stream_data_object()` method to stream a file from S3-backed HydroShare storage directly into a native Python object - in this case, loading a Zarr store into an `xarray.Dataset` - without downloading it first.

## Install the hsclient Python Client

The hsclient Python Client for HydroShare may not be installed by default in your Python environment, so it has to be installed first before you can work with it. Use the following command to install hsclient via the Python Package Index (PyPi). This will install the hsclient as well as all the python packages to work with HydroShare resource data as data processing python objects. The following packages will be installed in addition to hsclient:

* pandas
* fiona
* rasterio
* xarray
* netCDF4
* zarr
* h5netcdf
* pyarrow

In [ ]:
!pip install hsclient[all]

## 1. Connecting to HydroShare

Sign in to HydroShare and load the resource that contains the Zarr store we're interested in.

In [ ]:
import hsclient
# Sign in to HydroShare using the hsclient library
# This will prompt you to enter your username and password
hs = hsclient.HydroShare()
hs.sign_in()

## 2. Streaming Data Directly Into a Native Python Object

Rather than downloading a file and then loading it with a science library, `hsclient` can stream certain file formats directly from S3-backed HydroShare storage into a native Python object (e.g. an `xarray.Dataset`, `pandas.DataFrame`, or `rasterio` dataset) via `HydroShare.stream_data_object()`. This avoids a separate download step and allows libraries like `xarray`/`zarr` to lazily read only the chunks they need.

Below we stream a Zarr store (e.g., `zarr/sample_streamflow.zarr`) in this resource directly into an `xarray.Dataset`.

In [ ]:
# Build the fully-qualified S3 path to the zarr store using the resource's S3 path
# Replace with the ID of a resource that you have access to and that contains a zarr store
resource_id = '863fc70b7f824b109b4f610a9f85f3a7'
resource_s3_path = hs.get_resource_path(resource_id)
# Replace with the relative path to your own zarr store within the resource
zarr_s3_path = f"{resource_s3_path}/zarr/sample_streamflow.zarr"
print(zarr_s3_path)

In [ ]:
# Stream the zarr store directly into an xarray.Dataset (requires the 'xarray' and 'zarr' extras)
ds = hs.stream_data_object(zarr_s3_path)
ds

Since `xarray.open_zarr()` lazily loads data by default, only the metadata/coordinates are read at this point - the actual data variables are fetched from S3 on demand (e.g. when you access `.values`, plot, or compute on them).

In [ ]:
# Inspect variables, dimensions, and coordinates without pulling the full dataset into memory
print(ds.data_vars)
print(ds.coords)

In [ ]:
# Accessing a variable's values triggers the actual data fetch from S3
var_name = list(ds.data_vars)[0]
ds[var_name].values